# 01 — Analyse Exploratoire des Données (EDA)

Visualisations des capteurs ESP32 : humidité sol, température, humidité air.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))  # racine du projet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# Style global
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

FAKE_DATA_PATH = "../../data/raw/fake_mesures.json"


## 1. Chargement des données

In [ ]:
from ml.src.data.loader import load_combined
from ml.src.data.cleaner import clean
from ml.src.data.feature_builder import build_features

df_raw = load_combined(fake_path=FAKE_DATA_PATH, prefer_api=False)
print(f"Shape brut : {df_raw.shape}")
df_raw.head(3)


## 2. Nettoyage

In [ ]:
df_clean = clean(df_raw)
print(df_clean.dtypes)
df_clean.describe().round(2)


## 3. Distribution des 3 features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
features = [
    ("humidite_sol", "Humidité Sol (%)", "#4e79a7"),
    ("temperature",  "Température (°C)", "#f28e2b"),
    ("humidite_air", "Humidité Air (%)", "#76b7b2"),
]
for ax, (col, label, color) in zip(axes, features):
    ax.hist(df_clean[col], bins=40, color=color, alpha=0.85, edgecolor="white")
    ax.axvline(df_clean[col].mean(), color="red", linestyle="--", label=f"Moy. {df_clean[col].mean():.1f}")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel(label)
    ax.legend()

plt.suptitle("Distribution des capteurs ESP32", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../ml/reports/figures/01_distributions.png", bbox_inches="tight")
plt.show()


## 4. Évolution temporelle sur 7 jours

In [ ]:
df_week = df_clean[df_clean["timestamp"] <= df_clean["timestamp"].iloc[0] + pd.Timedelta(days=7)]

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

axes[0].plot(df_week["timestamp"], df_week["humidite_sol"], color="#4e79a7", lw=1)
axes[0].axhline(20, color="red", linestyle="--", alpha=0.7, label="Seuil sec (20%)")
axes[0].axhline(50, color="orange", linestyle="--", alpha=0.7, label="Seuil normal (50%)")
axes[0].set_ylabel("Humidité Sol (%)")
axes[0].legend(loc="upper right")
axes[0].set_title("Humidité du Sol", fontweight="bold")

axes[1].plot(df_week["timestamp"], df_week["temperature"], color="#f28e2b", lw=1)
axes[1].set_ylabel("Température (°C)")
axes[1].set_title("Température", fontweight="bold")

axes[2].plot(df_week["timestamp"], df_week["humidite_air"], color="#76b7b2", lw=1)
axes[2].set_ylabel("Humidité Air (%)")
axes[2].set_title("Humidité de l'Air", fontweight="bold")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m %Hh"))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30)

plt.suptitle("Évolution sur 7 jours — Données ESP32", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/02_time_series.png", bbox_inches="tight")
plt.show()


## 5. Matrice de corrélation

In [ ]:
corr_cols = ["humidite_sol", "temperature", "humidite_air"]
corr = df_clean[corr_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, square=True,
            annot_kws={"size": 13})
plt.title("Corrélation entre les features", fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/03_correlation.png", bbox_inches="tight")
plt.show()


## 6. Distribution des états du sol (labels)

In [ ]:
from ml.src.data.feature_builder import build_features
df = build_features(df_clean)

counts = df["etat_sol"].value_counts()
colors = {"sec": "#e15759", "normal": "#59a14f", "humide": "#4e79a7"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barres
bars = axes[0].bar(counts.index, counts.values,
                   color=[colors[c] for c in counts.index], edgecolor="white")
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f"{val}", ha="center", fontweight="bold")
axes[0].set_title("Nombre de mesures par état du sol", fontweight="bold")
axes[0].set_ylabel("Nombre de mesures")

# Pie
axes[1].pie(counts.values, labels=counts.index,
            colors=[colors[c] for c in counts.index],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Répartition des états", fontweight="bold")

plt.suptitle("Labels de classification — État du Sol", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/04_class_distribution.png", bbox_inches="tight")
plt.show()

print(f"\nBesoin d'eau (True) : {df['besoin_eau'].sum()} / {len(df)} ({df['besoin_eau'].mean()*100:.1f}%)")


## 7. Humidité sol selon l'heure de la journée

In [ ]:
df["hour_int"] = df["timestamp"].dt.hour
hourly = df.groupby("hour_int")[["humidite_sol", "temperature"]].mean()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(hourly.index, hourly["humidite_sol"], color="#4e79a7", alpha=0.6, label="Humidité Sol moy.")
ax2.plot(hourly.index, hourly["temperature"], color="#f28e2b", lw=2.5, marker="o", label="Température moy.")

ax1.set_xlabel("Heure de la journée")
ax1.set_ylabel("Humidité Sol (%)", color="#4e79a7")
ax2.set_ylabel("Température (°C)", color="#f28e2b")
ax1.set_xticks(range(24))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Humidité sol & température par heure", fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/05_hourly_pattern.png", bbox_inches="tight")
plt.show()
